# 12 - Verify the Cache

Step 1 reduces 570 GB of DICOM to a 29 GB cache and everything downstream
reads only the cache. Its correctness has never been checked against the
shards themselves: the local suite proves the *functions* are right, and the
manifests prove the *bookkeeping* is right, but nothing has confirmed that the
pixels in the shipped shards are what today's code produces.

The shards were built 2026-08-23. `src/data/` changed afterwards. Every number
this project has measured, the 0.906 included, was measured on these volumes.

**Attach, in the sidebar:**

| input | what for |
|---|---|
| competition `rsna-knee-abnormality-detection` | raw DICOM, for the rebuild |
| notebook output `knee-cache-224-01-shard-1` | shard 0 + its manifest |
| notebook output `knee-cache-224-01-shard-2` | shard 1 + its manifest |
| notebook output `knee-cache-224-01-shard-3` | shard 2 + its manifest |
| dataset `pssv27/knee-index` *(optional)* | header index, for the orientation audit |

**Accelerator: None.** No GPU is used, so this costs nothing against the
25 GPU-hour weekly quota. Internet is not needed.

Everything runs from `tools/verify_cache.py` and `tools/audit_orientation.py`,
embedded below from the repo rather than reimplemented -- reimplementing a
preprocessing step in a notebook is how a train/test skew gets introduced.

In [ ]:
# Materialise the repo modules. Generated from source by
# tools/make_notebook.py -- do not edit here, edit src/ and regenerate.
import importlib, pathlib, sys

FILES = {
    'src/__init__.py': '',
    'src/config.py': '"""Central configuration: label names, cache geometry, and environment paths.\n\nEverything here is imported by both the training pipeline and the submission\nnotebook, so it must stay dependency-free apart from the standard library.\n"""\n\nimport os\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Labels\n# ---------------------------------------------------------------------------\n# Exact submission spelling. The apostrophe in "Baker\'s" and the spaces are\n# part of the column names -- do not normalise them anywhere that touches a\n# submission file.\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\n\nID_COL = "StudyInstanceUID"\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\n\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\n# any augmentation that mirrors the image must also swap these pairs.\nLATERAL_LABEL_PAIRS = [\n    ("Medial Meniscus", "Lateral Meniscus"),\n    ("Medial OA", "Lateral OA"),\n]\n\n# ---------------------------------------------------------------------------\n# Cache geometry\n# ---------------------------------------------------------------------------\n# Kept as constants so a higher-resolution variant is a config change rather\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\n# x 24,371 series ~= 30 GB, which fits Kaggle\'s 200 GB private dataset limit.\nCACHE_SIZE = 224          # in-plane resolution after resampling\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\nCACHE_DTYPE = "uint8"\n\n# Robust intensity normalisation percentiles. MRI has no standardised units,\n# so every volume is scaled against its own distribution.\nCLIP_PCT_LOW = 1.0\nCLIP_PCT_HIGH = 99.0\n\n# Canonical laterality: left knees are mirrored to match right knees so the\n# medial compartment always falls on the same side of the image.\nCANONICAL_SIDE = "R"\n\n# Mirroring is applied at load time, not baked into the cache.\n#\n# The side tag is blank on most series in this dataset and the series\n# descriptions are anonymised, so roughly 42% of series have no recoverable\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\n# would mean rebuilding all of it every time side detection improves -- and it\n# is expected to improve, via study-level propagation and then a classifier\n# trained on the series whose side is known. The cache therefore stores volumes\n# in their acquired orientation with the side recorded alongside, and the data\n# loader flips on read, which costs nothing.\nMIRROR_IN_CACHE = False\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\nCOMPETITION = "rsna-knee-abnormality-detection"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\n# What a competition data root must contain to be recognised.\n_MARKERS = ("train_series", "test_series")\n\n\ndef _detect_data_root() -> Path:\n    """Locate the competition data across Kaggle, Colab, and local runs.\n\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\'s input directory is\n    searched by content rather than by name: the mount is not reliably at\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\n    through to the local fallback, which then fails much later with a confusing\n    path in the message.\n    """\n    env = os.environ.get("KNEE_DATA_ROOT")\n    if env:\n        return Path(env)\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        exact = kaggle_input / COMPETITION\n        if exact.exists():\n            return exact\n        # Search two levels: Kaggle mounts competition data under\n        # /kaggle/input/competitions/<slug>/ as well as directly under\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\n        try:\n            for child in sorted(kaggle_input.iterdir()):\n                if not child.is_dir():\n                    continue\n                if any((child / m).exists() for m in _MARKERS):\n                    return child\n                for grandchild in sorted(child.iterdir()):\n                    if grandchild.is_dir() and any(\n                            (grandchild / m).exists() for m in _MARKERS):\n                        return grandchild\n        except OSError:\n            pass\n\n    return PROJECT_ROOT / "data"\n\n\nDATA_ROOT = _detect_data_root()\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\n\nTRAIN_CSV = META_DIR / "train.csv"\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\nTEST_CSV = META_DIR / "test.csv"\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\n\nINDEX_PARQUET = META_DIR / "index.parquet"\n\nON_KAGGLE = Path("/kaggle").exists()\n',
    'src/data/__init__.py': '',
    'src/data/dicom_io.py': '"""DICOM reading, geometric slice ordering, and laterality resolution.\n\nTwo operations here are load-bearing and fail silently if done naively:\n\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\n   slice normal derived from ImageOrientationPatient, which is defined by the\n   scanner geometry rather than by convention.\n\n2. **Laterality.** Four of the twelve labels are compartment-specific\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\n   on the opposite side of the frame from a right knee, so left knees must be\n   mirrored into a canonical orientation before a model can learn the\n   distinction.\n"""\n\nimport re\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport pydicom\n\n# Header fields pulled during the index scan. Kept explicit so the parquet\n# schema is stable and the scan stays header-only.\nHEADER_FIELDS = [\n    "StudyInstanceUID",\n    "SeriesInstanceUID",\n    "SOPInstanceUID",\n    "Rows",\n    "Columns",\n    "PixelSpacing",\n    "ImagePositionPatient",\n    "ImageOrientationPatient",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "ScanningSequence",\n    "SequenceVariant",\n    # (0018,0010). The one header that states contrast administration\n    # outright, and the index built before 2026-08-31 does not carry it --\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\n    # which are anonymised away. Absence of a string is not absence of\n    # contrast; absence of this tag is at least a direct question asked.\n    "ContrastBolusAgent",\n    "SeriesDescription",\n    "ProtocolName",\n    "BodyPartExamined",\n    "Manufacturer",\n    "ManufacturerModelName",\n    "MagneticFieldStrength",\n    "ImageLaterality",\n    "Laterality",\n    "InstanceNumber",\n]\n\n# DICOM DS/IS values carry their original string representation, which would\n# land in the index as object dtype. Cast these explicitly.\nNUMERIC_FIELDS = {\n    "Rows",\n    "Columns",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "MagneticFieldStrength",\n    "InstanceNumber",\n}\n\n_LEFT_PATTERN = re.compile(\n    r"\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\b",\n    re.IGNORECASE,\n)\n_RIGHT_PATTERN = re.compile(\n    r"\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\b",\n    re.IGNORECASE,\n)\n\n\ndef read_header(path) -> dict:\n    """Read one DICOM\'s headers without decoding pixel data."""\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\n    row = {"path": str(path)}\n    for field in HEADER_FIELDS:\n        value = getattr(ds, field, None)\n        if value is None:\n            row[field] = None\n        elif isinstance(value, pydicom.multival.MultiValue):\n            # Geometry fields are numeric; ScanningSequence and friends are\n            # multi-valued strings, so fall back to a joined token.\n            try:\n                row[field] = [float(v) for v in value]\n            except (TypeError, ValueError):\n                row[field] = "\\\\".join(str(v) for v in value)\n        elif field in NUMERIC_FIELDS:\n            try:\n                row[field] = float(value)\n            except (TypeError, ValueError):\n                row[field] = None\n        elif isinstance(value, (int, float, str)):\n            row[field] = str(value) if isinstance(value, str) else value\n        else:\n            row[field] = str(value)\n    return row\n\n\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\n    """Unit normal of the image plane, from the two direction cosines.\n\n    Falls back to the z axis when orientation is missing, which keeps a\n    malformed series ordered by patient-superior position rather than crashing.\n    """\n    if orientation is None or len(orientation) != 6:\n        return np.array([0.0, 0.0, 1.0])\n\n    row_cosine = np.asarray(orientation[:3], dtype=float)\n    col_cosine = np.asarray(orientation[3:], dtype=float)\n    normal = np.cross(row_cosine, col_cosine)\n\n    magnitude = np.linalg.norm(normal)\n    if magnitude == 0:\n        return np.array([0.0, 0.0, 1.0])\n    return normal / magnitude\n\n\ndef slice_position(header: dict) -> float:\n    """Scalar position of a slice along its own stack axis."""\n    position = header.get("ImagePositionPatient")\n    if position is None or len(position) != 3:\n        # No geometry available; fall back to InstanceNumber so ordering is at\n        # least deterministic. The index scan flags these series separately.\n        number = header.get("InstanceNumber")\n        return float(number) if number is not None else 0.0\n\n    normal = slice_normal(header.get("ImageOrientationPatient"))\n    return float(np.dot(np.asarray(position, dtype=float), normal))\n\n\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\n    """Order a series\' slices along the acquisition axis."""\n    return sorted(headers, key=slice_position)\n\n\ndef has_geometry(headers: Sequence[dict]) -> bool:\n    """True when every slice carries the position and orientation tags."""\n    for header in headers:\n        position = header.get("ImagePositionPatient")\n        orientation = header.get("ImageOrientationPatient")\n        if position is None or len(position) != 3:\n            return False\n        if orientation is None or len(orientation) != 6:\n            return False\n    return True\n\n\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\n    """Determine whether a series images a left or right knee.\n\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\n    description fields, which carry side markers in several languages. Returns\n    "L", "R", or None when the side cannot be established -- callers must treat\n    None as unusable rather than guessing, since a wrong side silently swaps the\n    medial and lateral labels.\n    """\n    for header in headers:\n        for field in ("ImageLaterality", "Laterality"):\n            value = header.get(field)\n            if isinstance(value, str):\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\n                # often present-but-empty than absent. An empty string is not a\n                # side and must not be read as one.\n                token = value.strip().upper()\n                if token in ("L", "LEFT"):\n                    return "L"\n                if token in ("R", "RIGHT"):\n                    return "R"\n\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\n        for header in headers:\n            text = header.get(field)\n            if not isinstance(text, str):\n                continue\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\n            left = bool(_LEFT_PATTERN.search(spaced))\n            right = bool(_RIGHT_PATTERN.search(spaced))\n            # Ambiguous strings mentioning both sides tell us nothing.\n            if left and not right:\n                return "L"\n            if right and not left:\n                return "R"\n\n    return None\n\n\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\n\n    Applies the rescale slope and intercept when present. Slices whose in-plane\n    shape differs from the first are skipped rather than silently broadcast.\n    """\n    slices = []\n    reference_shape = None\n    skipped = 0\n\n    for path in paths:\n        try:\n            ds = pydicom.dcmread(str(path), force=True)\n            pixels = ds.pixel_array.astype(np.float32)\n        except Exception:  # noqa: BLE001\n            # A truncated or malformed file loses one slice, not the series.\n            # The dataset contains files whose declared pixel length exceeds\n            # what is actually stored, and dropping 23 good slices over one\n            # bad one costs a whole study its imaging.\n            skipped += 1\n            continue\n\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n        if slope != 1.0 or intercept != 0.0:\n            pixels = pixels * slope + intercept\n\n        if reference_shape is None:\n            reference_shape = pixels.shape\n        elif pixels.shape != reference_shape:\n            skipped += 1\n            continue\n\n        slices.append(pixels)\n\n    if not slices:\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\n\n    return np.stack(slices, axis=0)\n\n\ndef series_paths(series_dir) -> List[str]:\n    """All DICOM files under one series directory."""\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\n\n\ndef resolve_study_laterality(series_sides: dict) -> dict:\n    """Fill in unknown series sides from other series in the same study.\n\n    A knee MRI study images one knee, so a side established on any series\n    applies to all of them. This matters because the DICOM side tags are\n    populated inconsistently across the contributing institutions -- a study\n    can carry the tag on its coronal series and omit it on the sagittals.\n\n    Report text is deliberately not used as a source here. Reports exist only\n    for training studies, so recovering a side from them would mirror the\n    training volumes and leave the test volumes untouched -- a train/test skew\n    that is worse than the missing side it fixes.\n\n    Args:\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\n\n    Returns:\n        The same structure with unknown entries filled where the study allows.\n        Studies whose series disagree are left untouched, since a conflict\n        means the assumption of one knee per study does not hold there.\n    """\n    resolved = {}\n    for study_uid, series in series_sides.items():\n        known = {side for side in series.values() if side in ("L", "R")}\n\n        if len(known) == 1:\n            side = known.pop()\n            resolved[study_uid] = {uid: side for uid in series}\n        else:\n            # No evidence, or contradictory evidence. Guessing would silently\n            # swap the medial and lateral labels on four of twelve findings.\n            resolved[study_uid] = dict(series)\n\n    return resolved\n\n\ndef laterality_coverage(series_sides: dict) -> dict:\n    """Counts before and after study-level propagation, for auditing."""\n    before = sum(1 for s in series_sides.values() for v in s.values()\n                 if v in ("L", "R"))\n    total = sum(len(s) for s in series_sides.values())\n\n    filled = resolve_study_laterality(series_sides)\n    after = sum(1 for s in filled.values() for v in s.values()\n                if v in ("L", "R"))\n\n    conflicts = sum(\n        1 for s in series_sides.values()\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\n    )\n\n    return {\n        "total_series": total,\n        "known_before": before,\n        "known_after": after,\n        "recovered": after - before,\n        "conflicting_studies": conflicts,\n    }\n',
    'src/data/cache.py': '"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\n\nThe same code path runs during cache construction and at submission time. Any\ndivergence between the two produces a train/test preprocessing skew, which is\ninvisible in local validation and only shows up as a collapsed leaderboard\nscore -- so callers must import from here rather than reimplementing.\n\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\norientation it was acquired in. Mirroring to a canonical side happens at load\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport cv2\nimport numpy as np\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\n    """Scale a volume to uint8 using its own robust percentile range.\n\n    MRI carries no standardised units -- intensity depends on scanner, field\n    strength, coil, and whether fat suppression was applied -- so each volume is\n    scaled against its own distribution rather than a global constant.\n    """\n    finite = volume[np.isfinite(volume)]\n    if finite.size == 0:\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\n        # Flat or degenerate volume; nothing to stretch.\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\n    return (scaled * 255.0).round().astype(np.uint8)\n\n\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\n    """Resample every slice to size x size with area/linear interpolation."""\n    size = size or config.CACHE_SIZE\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\n\n    for i, plane in enumerate(volume):\n        # Area interpolation avoids the aliasing that bilinear introduces when\n        # downsampling 800px acquisitions; linear is correct when upsampling.\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\n\n    return output\n\n\ndef select_slices(count: int, target: int = None) -> List[int]:\n    """Indices of `target` slices spread evenly across a stack of `count`.\n\n    Short series are handled by repeating edge slices rather than zero-padding,\n    so the volume stays anatomically continuous.\n    """\n    target = target or config.CACHE_SLICES\n    if count <= 0:\n        raise ValueError("cannot select slices from an empty series")\n    if count == target:\n        return list(range(count))\n\n    positions = np.linspace(0, count - 1, target)\n    return [int(round(p)) for p in positions]\n\n\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\n\n    Mirroring a knee means reflecting through the patient\'s left-right axis,\n    and which array axis that corresponds to depends on the acquisition plane:\n\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\n\n    A blanket column flip is wrong for sagittal series. It leaves medial and\n    lateral exactly where they were -- they are separated across slices, not\n    across the image -- and instead reflects anterior-posterior, moving the\n    patella to the back of the knee.\n\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\n    """\n    left_right = np.array([1.0, 0.0, 0.0])\n\n    if orientation is None or len(orientation) != 6:\n        # No geometry to reason from. Columns is the common case (coronal and\n        # axial together outnumber sagittal), but the caller should prefer not\n        # to mirror at all when geometry is missing.\n        return 2\n\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\n    normal = dicom_io.slice_normal(orientation)\n\n    alignments = [\n        (abs(float(normal @ left_right)), 0),\n        (abs(float(height @ left_right)), 1),\n        (abs(float(width @ left_right)), 2),\n    ]\n    return max(alignments)[1]\n\n\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\n    """Mirror left knees so the medial compartment is always on one side.\n\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\n    Lateral OA). Without this the model would have to infer the side before it\n    could place a finding, from a cue that is not reliably in the image.\n\n    The axis reflected is chosen from the acquisition geometry -- see\n    mirror_axis -- because it differs between sagittal and coronal series.\n    """\n    if side is None:\n        # Unknown side: leave untouched and let the caller decide. Guessing\n        # would silently swap four of the twelve labels.\n        return volume\n    if side.upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = mirror_axis(orientation)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef submission_indices(count: int, slices: int) -> List[int]:\n    """Raw slice indices a `slices`-deep submission actually reads.\n\n    **Not** `select_slices(count, slices)`, and the difference is the whole\n    reason this exists. The submission decodes `CACHE_SLICES` and then\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\n    `slices` evenly across the raw stack in one step lands on *different\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\n    then be scored on anatomy it was not trained to read, and nothing would\n    raise -- the shapes agree either way. That is the same failure the\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\n\n    So the two selections are composed rather than collapsed: the coarse pick\n    the cache would have made, then the fine pick the model applies to it.\n    """\n    coarse = select_slices(count)\n    if slices == config.CACHE_SLICES:\n        return coarse\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\n\n\ndef build_series_volume(headers: Sequence[dict],\n                        slices: Optional[int] = None,\n                        size: Optional[int] = None) -> Dict:\n    """Full pipeline for one series: sort, decode, normalise, resize.\n\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\n    not by default. The recorded `laterality` is what the loader mirrors on.\n    Returns the volume plus the provenance needed to audit it later.\n\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\n    decoding done for images the model never sees, and inference time is a\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\n    construction is unchanged and every existing shard stays valid.\n\n    **It is not bit-identical to decoding the full depth.**\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\n    computed over whatever was decoded, so a shallower stack shifts the\n    intensity mapping slightly. The slices themselves are the same files --\n    see `submission_indices` -- but the uint8 values differ a little, and the\n    submission notebook measures that difference rather than assuming it away.\n    """\n    if not headers:\n        raise ValueError("no headers supplied")\n    if size is not None and (not isinstance(size, int) or isinstance(size, bool) or size < 1):\n        raise ValueError("size must be a positive integer")\n\n    depth = config.CACHE_SLICES if slices is None else slices\n    ordered = dicom_io.sort_slices(headers)\n    first_header = ordered[0]\n    side = dicom_io.resolve_laterality(ordered)\n    geometry_ok = dicom_io.has_geometry(ordered)\n\n    indices = submission_indices(len(ordered), depth)\n    selected = [ordered[i] for i in indices]\n\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\n\n    # Slices skipped as unreadable leave the volume short of the requested\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\n    # decoded so the depth is exact regardless of how many files were dropped.\n    if raw.shape[0] != depth:\n        raw = raw[select_slices(raw.shape[0], depth)]\n\n    resized = resize_volume(raw.astype(np.float32), size=size)\n    normalized = normalize_intensity(resized)\n\n    # Left as acquired by default; the loader mirrors using the recorded side.\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\n    volume = (canonicalize_side(normalized, side,\n                                first_header.get("ImageOrientationPatient"))\n              if config.MIRROR_IN_CACHE else normalized)\n\n    first = ordered[0]\n    return {\n        "volume": volume,\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\n        "n_slices_raw": len(ordered),\n        "laterality": side,\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\n                         and side.upper() != config.CANONICAL_SIDE),\n        "geometry_ok": geometry_ok,\n        "rows": int(first.get("Rows") or 0),\n        "cols": int(first.get("Columns") or 0),\n    }\n\n\ndef build_series_record(item):\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\n\n    Defined at module level deliberately. A worker defined inside a notebook\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\n    spawn start method and survives under fork only by accident. Keeping it\n    importable makes the cache build work the same way everywhere.\n\n    Failures are returned rather than raised: one unreadable series must not\n    end a ninety-minute run, and the manifest records what to investigate.\n    """\n    series_uid, headers = item\n    try:\n        result = build_series_volume(headers)\n        return series_uid, result["volume"], {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": result["StudyInstanceUID"],\n            "n_slices_raw": result["n_slices_raw"],\n            "laterality": result["laterality"],\n            "mirrored": result["mirrored"],\n            "geometry_ok": result["geometry_ok"],\n            "rows": result["rows"],\n            "cols": result["cols"],\n            "error": None,\n        }\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\n        return series_uid, None, {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": study_uid,\n            "n_slices_raw": len(headers) if headers else 0,\n            "laterality": None,\n            "mirrored": False,\n            "geometry_ok": False,\n            "rows": 0,\n            "cols": 0,\n            "error": str(exc)[:200],\n        }\n',
    'src/data/index.py': '"""Header-only scan of the DICOM tree into a single parquet index.\n\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\nthe cost is dominated by file-system traversal rather than decoding. The\nresulting index drives cache construction and lets series be filtered and\naudited without touching the 570 GB of imaging again.\n\nUsage (in a Kaggle notebook):\n\n    from src.data.index import build_index\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\n"""\n\nimport argparse\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterator, List, Optional\n\nimport pandas as pd\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef iter_dicom_paths(root) -> Iterator[Path]:\n    """Yield every .dcm under root/study/series/ without materialising a list."""\n    root = Path(root)\n    for study_dir in sorted(root.iterdir()):\n        if not study_dir.is_dir():\n            continue\n        for series_dir in sorted(study_dir.iterdir()):\n            if not series_dir.is_dir():\n                continue\n            for path in sorted(series_dir.glob("*.dcm")):\n                yield path\n\n\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\n    """Read headers for a batch of files in parallel.\n\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\n    released during file reads and a process pool would only add overhead.\n    """\n    def safe_read(path):\n        try:\n            return dicom_io.read_header(path)\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\n            return {"path": str(path), "error": str(exc)}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(safe_read, paths))\n\n    return pd.DataFrame(rows)\n\n\ndef build_index(\n    split: str = "train_series",\n    out_path: Optional[str] = None,\n    data_root=None,\n    workers: int = 8,\n    batch_size: int = 5000,\n    limit: Optional[int] = None,\n) -> pd.DataFrame:\n    """Scan one split\'s DICOM tree and write a parquet index.\n\n    Processes in batches so a long scan reports progress and so peak memory\n    stays flat regardless of how many files the split holds.\n    """\n    root = Path(data_root or config.DATA_ROOT) / split\n    if not root.exists():\n        raise FileNotFoundError(f"{root} does not exist")\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:  # progress is a convenience, never a requirement\n        tqdm = None\n\n    frames = []\n    batch: List[Path] = []\n    scanned = 0\n\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\n    # counts files without a total rather than forcing a full listing first,\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\n\n    for path in iter_dicom_paths(root):\n        batch.append(path)\n        if limit is not None and len(batch) + scanned >= limit:\n            break\n        if len(batch) >= batch_size:\n            frames.append(scan_paths(batch, workers=workers))\n            scanned += len(batch)\n            if progress is not None:\n                progress.update(len(batch))\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\n            else:\n                print(f"  scanned {scanned} files", flush=True)\n            batch = []\n\n    if batch:\n        frames.append(scan_paths(batch, workers=workers))\n        scanned += len(batch)\n        if progress is not None:\n            progress.update(len(batch))\n\n    if progress is not None:\n        progress.close()\n\n    if not frames:\n        raise RuntimeError(f"no DICOM files found under {root}")\n\n    index = pd.concat(frames, ignore_index=True)\n    print(f"scanned {scanned} files across {index[\'SeriesInstanceUID\'].nunique()} series")\n\n    if "error" in index.columns:\n        failures = int(index["error"].notna().sum())\n        if failures:\n            print(f"WARNING: {failures} files failed to parse")\n\n    if out_path:\n        os.makedirs(Path(out_path).parent, exist_ok=True)\n        index.to_parquet(out_path, index=False)\n        print(f"wrote {out_path}")\n\n    return index\n\n\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\n    """Per-series roll-up: slice count, geometry completeness, laterality.\n\n    Surfaces the series that need attention -- missing geometry, unresolvable\n    side -- before an hour of cache construction is spent on them.\n    """\n    records = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        records.append({\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "SeriesInstanceUID": series_uid,\n            "n_slices": len(headers),\n            "geometry_ok": dicom_io.has_geometry(headers),\n            "laterality": dicom_io.resolve_laterality(headers),\n            "rows": headers[0].get("Rows"),\n            "cols": headers[0].get("Columns"),\n        })\n\n    return pd.DataFrame(records)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--out", default=None)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--limit", type=int, default=None)\n    args = parser.parse_args()\n\n    build_index(split=args.split, out_path=args.out,\n                workers=args.workers, limit=args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/model/__init__.py': '',
    'src/model/dataset.py': '"""Study-level dataset over the cached volumes.\n\nJoins four sources into one training sample per study:\n\n    cache shards      the uint8 volumes, memory-mapped\n    manifest          which shard row holds which series\n    fused labels      12 soft targets and 12 sample weights per study\n    side predictions  L/R per series, from the tag or the classifier\n\nA sample is a whole study, not a series. The twelve findings are study-level\nlabels, and a single series often cannot support them -- effusion reads off a\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\nhas to see the study\'s series together.\n\n**Laterality is handled two ways, depending on the path.**\n\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\nside at load time, along the axis the acquisition plane dictates: slice order\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\nand axial, where it lies in-plane. Doing it at load time rather than in the\ncache means side detection can improve without rebuilding 29 GB.\n\nFor the frozen path this is not available -- the embeddings are computed from\nthe volume, so the image can no longer be flipped. There the side travels to\nthe model as a covariate (`is_right`, `side_known`) instead. That still\naddresses the original concern, which was that the model should not have to\n*infer* the side from cues that are not reliably present; it is handed the\nanswer either way. Which of the two performs better is an experiment, not a\nsettled question.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\nPLANE_MIRROR_AXIS = {\n    "Sagittal": 0,   # medial-lateral is the slice axis\n    "Coronal": 2,    # medial-lateral lies in-plane\n    "Axial": 2,\n}\nDEFAULT_MIRROR_AXIS = 2\n\n# Series metadata fed to the model as conditioning. Findings are\n# sequence-specific, and these three are given in both train and test.\n# `is_right` and `side_known` carry laterality as a covariate rather than\n# normalising it away by mirroring. With precomputed frozen embeddings the\n# image can no longer be flipped -- the features are already computed -- so the\n# side is handed to the model instead, which addresses the original concern\n# (the model should not have to *infer* the side) by a different route.\n# Mirroring remains available for the end-to-end path and is an A/B to run,\n# not a settled question.\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\n                "is_coronal", "is_axial", "is_right", "side_known"]\n\n\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\n                  plane: Optional[str]) -> np.ndarray:\n    """Reflect a left knee onto the canonical side, along the correct axis."""\n    if not isinstance(side, str) or side.strip().upper() not in ("L", "R"):\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\n        # on four of the twelve findings.\n        return volume\n    if side.strip().upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef series_metadata(row: pd.Series) -> np.ndarray:\n    """Conditioning vector for one series.\n\n    Fluid sensitivity and fat suppression are kept separate: the data\n    description notes they are often correlated but not equivalent, so\n    collapsing them would discard the cases where they disagree.\n    """\n    plane = str(row.get("Anatomical_Plane", ""))\n    side = row.get("side_predicted")\n    side = side.strip().upper() if isinstance(side, str) else None\n    known = side in ("L", "R")\n    return np.array([\n        sequence_flag(row.get("Fluid_Sensitive")),\n        sequence_flag(row.get("Fat_Suppression")),\n        float(plane == "Sagittal"),\n        float(plane == "Coronal"),\n        float(plane == "Axial"),\n        float(known and side == "R"),\n        float(known),\n    ], dtype=np.float32)\n\n\ndef sequence_flag(value) -> float:\n    """Parse CSV flags without treating NaN or the string \'False\' as true."""\n    if value is None or pd.isna(value):\n        return 0.0\n    if isinstance(value, str):\n        value = value.strip().lower()\n        if value in ("true", "yes", "1", "1.0"):\n            return 1.0\n        if value in ("false", "no", "0", "0.0", ""):\n            return 0.0\n        raise ValueError(f"invalid sequence flag: {value!r}")\n    if value not in (0, 1):\n        raise ValueError(f"invalid sequence flag: {value!r}")\n    return float(value)\n\n\ndef select_series(rows: pd.DataFrame, maximum=None, strategy="uniform", rng=None):\n    """Positions to retain, preserving order and optionally protocol coverage.\n\n    Protocol selection first covers acquisition planes, then distinct\n    (plane, fluid sensitivity, fat suppression) tuples, then repeats. It\n    uses only metadata available at inference, never labels or reports.\n    """\n    if strategy not in ("uniform", "protocol"):\n        raise ValueError(f"unknown series selection: {strategy}")\n    if maximum is not None and maximum < 1:\n        raise ValueError("max_series must be positive or None")\n    count = len(rows)\n    if maximum is None or count <= maximum:\n        return np.arange(count)\n    if strategy == "uniform":\n        chosen = (np.linspace(0, count - 1, maximum).round().astype(int)\n                  if rng is None else rng.choice(count, maximum, replace=False))\n        return np.sort(chosen)\n    protocols = [(str(row.get("Anatomical_Plane", "")),\n                  sequence_flag(row.get("Fluid_Sensitive")),\n                  sequence_flag(row.get("Fat_Suppression")))\n                 for _, row in rows.iterrows()]\n    planes, covered, chosen = set(), set(), []\n    priority = (np.arange(count) if rng is None else rng.permutation(count))\n    while len(chosen) < maximum:\n        remaining = [int(i) for i in priority if i not in chosen]\n        best = max(remaining, key=lambda i: (\n            protocols[i][0] in PLANE_MIRROR_AXIS and protocols[i][0] not in planes,\n            protocols[i] not in covered,\n            min((abs(i - j) for j in chosen), default=0)))\n        chosen.append(best)\n        planes.add(protocols[best][0])\n        covered.add(protocols[best])\n    return np.array(sorted(chosen))\n\n\nclass StudyIndex:\n    """Everything needed to assemble a study, resolved once up front.\n\n    Built from frames rather than files so it can be constructed in a test\n    without touching disk.\n    """\n\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\n                 labels: Optional[pd.DataFrame] = None,\n                 sides: Optional[pd.DataFrame] = None,\n                 max_series: Optional[int] = None,\n                 reports: Optional[pd.DataFrame] = None,\n                 series_selection: str = "uniform"):\n        frame = manifest.merge(\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\n                         "Fat_Suppression", "Anatomical_Plane"]],\n            on="SeriesInstanceUID", how="left", validate="one_to_one")\n\n        if sides is not None:\n            frame = frame.merge(\n                sides[["SeriesInstanceUID", "side_predicted"]],\n                on="SeriesInstanceUID", how="left", validate="one_to_one")\n        else:\n            frame["side_predicted"] = frame.get("laterality_resolved")\n\n        # Drop series whose volume failed to build; their row in the cache is\n        # zeros and would train the model on blank images.\n        if "error" in frame.columns:\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\n\n        if series_selection == "protocol":\n            frame = frame.sort_values("SeriesInstanceUID", kind="stable").reset_index(drop=True)\n        self.series = frame\n        self.max_series = max_series\n        self.series_selection = series_selection\n        select_series(frame.iloc[:0], max_series, series_selection)\n        self.labels = labels\n\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\n        self._by_study = {\n            study: group.reset_index(drop=True)\n            for study, group in frame.groupby(config.ID_COL)\n        }\n\n        if labels is not None:\n            if labels[config.ID_COL].duplicated().any():\n                raise ValueError("duplicate study IDs in labels")\n            keyed = labels.set_index(config.ID_COL)\n            self.studies = [s for s in self.studies if s in keyed.index]\n            self._labels = keyed\n        else:\n            self._labels = None\n\n        # Report sentence embeddings, a training-time-only signal: reports\n        # exist for train studies and never for test, so these feed an\n        # auxiliary loss rather than the model input. Stored as one float32\n        # matrix rather than a DataFrame so __getitem__ is an array row.\n        if reports is not None:\n            keyed_reports = reports.set_index(config.ID_COL)\n            self._report_rows = {study: i for i, study\n                                 in enumerate(keyed_reports.index)}\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\n            self.report_dim = self._report_matrix.shape[1]\n        else:\n            self._report_rows = None\n            self._report_matrix = None\n            self.report_dim = None\n\n    def __len__(self) -> int:\n        return len(self.studies)\n\n    def series_for(self, study: str, rng=None) -> pd.DataFrame:\n        """Series of a study, capped deterministically when a cap is set.\n\n        The cap exists for the efficiency track: inference cost scales with\n        series count, and a study with a dozen series contributes far more\n        runtime than one with three. Selection is by position rather than at\n        random so a prediction is reproducible.\n\n        `rng` is the training-only exception. The deterministic cap discards\n        the same 3,013 of 24,371 series (12.4%) on every epoch of every fold --\n        1,432 studies carry more series than `max_series` and always show the\n        model the same subset. Passing a generator resamples the cap per epoch,\n        which reaches those series without adding a byte to the cache. Order is\n        preserved, so the series-axis metadata still lines up. Never pass one\n        for validation or inference: a prediction that changes between two\n        identical runs cannot be compared against anything.\n        """\n        rows = self._by_study[study]\n        keep = select_series(rows, self.max_series, self.series_selection, rng)\n        return rows.iloc[keep].reset_index(drop=True)\n\n    def targets_for(self, study: str):\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\n        if self._labels is None or study not in self._labels.index:\n            return None, None\n        row = self._labels.loc[study]\n        targets = np.array([float(row[name]) for name in config.LABELS],\n                           dtype=np.float32)\n        weights = np.array(\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\n            dtype=np.float32)\n        return targets, weights\n\n    def report_for(self, study: str) -> Optional[np.ndarray]:\n        """The study\'s report embedding, or None when unavailable."""\n        if self._report_rows is None or study not in self._report_rows:\n            return None\n        return self._report_matrix[self._report_rows[study]]\n\n\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray],\n               series_rng=None) -> Dict:\n    """Assemble one study: mirrored volumes, conditioning, targets, weights.\n\n    `series_rng` is forwarded to `series_for` and is training-only; see there.\n    """\n    rows = index.series_for(study, rng=series_rng)\n\n    volumes, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        volume = np.asarray(shard[int(row["position"])])\n        volume = mirror_volume(volume, row.get("side_predicted"),\n                               row.get("Anatomical_Plane"))\n        volumes.append(volume)\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(volumes) if volumes else np.zeros(\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\n            dtype=np.uint8),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef load_study_features(index: StudyIndex, study: str,\n                        shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study from precomputed slice embeddings.\n\n    Mirroring is unavailable here -- the features were computed from the image\n    and cannot be flipped after the fact -- so laterality reaches the model\n    through `series_metadata` instead.\n    """\n    rows = index.series_for(study)\n\n    features, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\n                                                                dtype=np.float32),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n        "report": index.report_for(study),\n    }\n\n\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\n    """Pad a batch of studies to the largest series count, with a mask.\n\n    Series counts vary from one to a dozen, so the batch is padded and masked\n    rather than truncated. The mask must reach the attention pooling: without\n    it, padding rows are averaged in as if they were real series.\n    """\n    import torch\n\n    longest = max(len(item["volumes"]) for item in batch)\n\n    # Per-series shape is taken from the data rather than assumed, so the same\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\n                     None)\n    if reference is None:\n        raise ValueError("every study in the batch is empty")\n    per_series = reference.shape[1:]\n    dtype = reference.dtype\n\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\n    mask = np.zeros((len(batch), longest), dtype=bool)\n\n    for i, item in enumerate(batch):\n        count = len(item["volumes"])\n        if count:\n            volumes[i, :count] = item["volumes"]\n            meta[i, :count] = item["meta"]\n            mask[i, :count] = True\n\n    out = {\n        config.ID_COL: [item[config.ID_COL] for item in batch],\n        "volumes": torch.from_numpy(volumes),\n        "meta": torch.from_numpy(meta),\n        "mask": torch.from_numpy(mask),\n    }\n\n    if batch[0]["targets"] is not None:\n        out["targets"] = torch.from_numpy(\n            np.stack([item["targets"] for item in batch]))\n        out["weights"] = torch.from_numpy(\n            np.stack([item["weights"] for item in batch]))\n\n    # Report embeddings, present only where the index carries them. Studies\n    # without one get a zero row and a False mask so the auxiliary loss can\n    # exclude them without a ragged batch.\n    reports = [item.get("report") for item in batch]\n    dims = [r.shape[0] for r in reports if r is not None]\n    if dims:\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\n        report_mask = np.zeros(len(batch), dtype=bool)\n        for i, row in enumerate(reports):\n            if row is not None:\n                report[i] = row\n                report_mask[i] = True\n        out["report"] = torch.from_numpy(report)\n        out["report_mask"] = torch.from_numpy(report_mask)\n    return out\n',
    'tools/verify_cache.py': '"""Verify the shipped cache against the code that claims to have built it.\n\nThe local test suite proves the step-1 *functions* are correct. It cannot prove\nthe 29 GB of shards on Kaggle are what those functions produce today -- the\nshards were built once, weeks before several changes to `src/data/`, and every\nfailure mode here is silent: a stale shard, a study split across shards, a\nvolume that normalised to all zeros, a series that failed to build and is now a\nblock of black images with a real label attached.\n\nRun on a **CPU** Kaggle kernel (no GPU hours) with the competition data and the\nthree cache shards attached:\n\n    !python tools/verify_cache.py --rebuild 40\n\nThe decisive check is `--rebuild`: it re-derives N random series from the raw\nDICOM with today\'s code and compares them byte for byte against the cached\nrows. Anything other than an exact match means the cache and the code have\ndiverged, and every number this project has measured was measured on the cache.\n"""\n\nimport argparse\nimport glob\nimport hashlib\nimport os\nimport random\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nfrom src import config\nfrom src.data import cache, dicom_io\n\nFAILURES = []\n\n\ndef check(name, ok, detail=""):\n    """Record and print one invariant. PASS means a failure mode is absent."""\n    if not ok:\n        FAILURES.append(name)\n    print(f"[{\'PASS\' if ok else \'FAIL\'}] {name}")\n    if detail:\n        for line in str(detail).splitlines():\n            print(f"         {line}")\n\n\ndef find_inputs(pattern):\n    """Cache shards land under /kaggle/input/<dataset>/ with unpredictable names.\n\n    Skips `/kaggle/input/competitions` exactly as notebook 08 does: a recursive\n    glob over the competition mount walks ~820,000 DICOM files for nothing.\n    """\n    hits = sorted(glob.glob(pattern))\n    hits += sorted(glob.glob(f"/kaggle/working/{pattern}"))\n    for root in sorted(glob.glob("/kaggle/input/*")):\n        if os.path.basename(root) == "competitions":\n            continue\n        hits += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))\n    return hits\n\n\ndef load_manifests(split):\n    """Every shard\'s manifest, tagged with the shard file when one is present.\n\n    Driven by the manifests rather than by the .npy files, because the\n    manifests answer most of this script\'s questions and are ~400 KB each\n    against ~10 GB. `kaggle kernels output <kernel> --file-pattern\n    \'manifest.*parquet\'` fetches them alone, so the inventory and sharding\n    checks run anywhere; only the volume and rebuild checks need the shards.\n\n    `cache_path` is NaN when the shard is absent, which is what the volume\n    checks test for before reading pixels.\n    """\n    frames = []\n    for manifest_path in find_inputs(f"manifest_{split}_*.parquet"):\n        frame = pd.read_parquet(manifest_path)\n        cache_path = manifest_path.replace("manifest_", "cache_").replace(".parquet", ".npy")\n        if not os.path.exists(cache_path):\n            stem = os.path.basename(manifest_path)[len("manifest_"):-len(".parquet")]\n            hits = [c for c in find_inputs(f"cache_{split}_*.npy")\n                    if os.path.basename(c).startswith(f"cache_{stem}")]\n            cache_path = hits[0] if hits else None\n        frame["cache_path"] = cache_path\n        frame["manifest_path"] = manifest_path\n        frames.append(frame)\n    if not frames:\n        raise SystemExit(f"no shard manifests found for split {split}")\n    return pd.concat(frames, ignore_index=True)\n\n\ndef check_inventory(manifest, expected_series, expected_studies):\n    n_series = manifest["SeriesInstanceUID"].nunique()\n    n_studies = manifest["StudyInstanceUID"].nunique()\n    n_shards = manifest["manifest_path"].nunique()\n\n    check("A1 all three shards are attached", n_shards == 3, f"{n_shards} shard(s) found")\n    check("A2 series count matches the documented cache",\n          n_series == expected_series,\n          f"{n_series} series (expected {expected_series})")\n    check("A3 study count matches the documented corpus",\n          n_studies == expected_studies,\n          f"{n_studies} studies (expected {expected_studies})")\n    check("A4 no duplicate series across shards",\n          len(manifest) == n_series,\n          f"{len(manifest)} rows for {n_series} distinct series")\n\n    if "error" in manifest.columns:\n        failed = manifest[manifest["error"].notna()]\n        check("A5 no series failed to build",\n              len(failed) == 0,\n              f"{len(failed)} failed; these rows are ZEROS in the cache and are "\n              f"dropped by StudyIndex -- confirm the drop actually happens\\n"\n              + failed[["SeriesInstanceUID", "error"]].head(10).to_string()\n              if len(failed) else "0 build errors recorded")\n\n    if "geometry_ok" in manifest.columns:\n        bad_geom = int((~manifest["geometry_ok"].astype(bool)).sum())\n        print(f"[INFO] {bad_geom} series lack complete geometry "\n              f"({100 * bad_geom / len(manifest):.2f}%) -- these sorted by "\n              f"InstanceNumber, which is unreliable")\n\n    if "laterality_resolved" in manifest.columns:\n        known = manifest["laterality_resolved"].isin(["L", "R"]).sum()\n        print(f"[INFO] laterality after study propagation: {known}/{len(manifest)} "\n              f"known ({100 * known / len(manifest):.1f}%), "\n              f"{100 * (1 - known / len(manifest)):.1f}% left to the classifier")\n\n    if "n_slices_raw" in manifest.columns:\n        raw = manifest["n_slices_raw"]\n        short = int((raw < config.CACHE_SLICES).sum())\n        print(f"[INFO] acquired depth: median {int(raw.median())}, "\n              f"range {int(raw.min())}-{int(raw.max())}; "\n              f"{short} series ({100 * short / len(manifest):.1f}%) are shorter "\n              f"than {config.CACHE_SLICES} and had slices repeated")\n\n\ndef check_sharding(manifest):\n    """Every series of a study must live in one shard, or study assembly is partial."""\n    per_study = manifest.groupby("StudyInstanceUID")["manifest_path"].nunique()\n    split = per_study[per_study > 1]\n    check("B1 no study is split across shards",\n          len(split) == 0,\n          f"{len(split)} studies span multiple shards -- those studies train on "\n          f"a subset of their own series\\n{split.head().to_string()}"\n          if len(split) else f"all {len(per_study)} studies are shard-local")\n\n    # The sharding rule itself, re-derived rather than trusted.\n    def shard_of(study_uid):\n        return int(hashlib.md5(str(study_uid).encode()).hexdigest(), 16) % 3\n\n    shard_index = {path: i for i, path in enumerate(sorted(manifest["manifest_path"].unique()))}\n    if len(shard_index) == 3:\n        actual = manifest["manifest_path"].map(shard_index)\n        expected = manifest["StudyInstanceUID"].map(shard_of)\n        agree = float((actual == expected).mean())\n        check("B2 shard assignment reproduces md5(study_uid) % 3",\n              agree > 0.999,\n              f"{100 * agree:.2f}% of series sit in the shard the hash predicts "\n              f"(shard files sorted by name; a low number may mean the file "\n              f"order differs from the SHARD constant, not that the rule broke)")\n\n\ndef check_volumes(manifest, sample):\n    """Find volumes that are blank or degenerate -- normalize_intensity\'s silent path."""\n    if manifest["cache_path"].isna().all():\n        print("[SKIP] C  cache shards not attached (manifests only); the blank- and\\n"\n              "         degenerate-volume checks need the .npy files")\n        return {}\n\n    rows = manifest[manifest["cache_path"].notna()]\n    rows = rows.sample(n=min(sample, len(rows)), random_state=0)\n\n    blank, low_var, stats = 0, 0, []\n    by_shard = {}\n    for path in rows["cache_path"].unique():\n        by_shard[path] = np.load(path, mmap_mode="r")\n\n    for _, row in rows.iterrows():\n        shard = by_shard[row["cache_path"]]\n        # One mid slice per series keeps this to ~50 KB of reads per volume.\n        plane = np.asarray(shard[int(row["position"])][config.CACHE_SLICES // 2])\n        if plane.max() == 0:\n            blank += 1\n        if plane.std() < 1.0:\n            low_var += 1\n        stats.append((float(plane.mean()), float(plane.std())))\n\n    means = np.array([s[0] for s in stats])\n    stds = np.array([s[1] for s in stats])\n    check(f"C1 no blank volumes in a {len(rows)}-series sample",\n          blank == 0, f"{blank} all-zero mid slices")\n    check(f"C2 no degenerate (near-constant) volumes in the sample",\n          low_var == 0, f"{low_var} mid slices with std < 1.0/255")\n    print(f"[INFO] sampled intensity: mean {means.mean():.1f}+/-{means.std():.1f}, "\n          f"per-slice std {stds.mean():.1f}+/-{stds.std():.1f} (uint8 scale)")\n    return by_shard\n\n\ndef check_rebuild(manifest, by_shard, count, split):\n    """The decisive check: does today\'s code reproduce the shipped bytes?"""\n    if not by_shard:\n        print("[SKIP] D  cache shards not attached; the byte-for-byte rebuild needs "\n              "them alongside the raw DICOM")\n        return\n\n    root = Path(config.DATA_ROOT) / split\n    if not root.exists():\n        print(f"[SKIP] D  competition data not attached ({root} missing); "\n              f"the rebuild check cannot run")\n        return\n\n    rows = manifest[manifest.get("error").isna()] if "error" in manifest.columns else manifest\n    rows = rows.sample(n=min(count, len(rows)), random_state=1)\n\n    exact, differing, missing = 0, [], 0\n    for _, row in rows.iterrows():\n        series_dir = root / str(row["StudyInstanceUID"]) / str(row["SeriesInstanceUID"])\n        paths = sorted(series_dir.glob("*.dcm"))\n        if not paths:\n            missing += 1\n            continue\n\n        headers = [dicom_io.read_header(p) for p in paths]\n        rebuilt = cache.build_series_volume(headers)["volume"]\n        cached = np.asarray(by_shard[row["cache_path"]][int(row["position"])])\n\n        if np.array_equal(rebuilt, cached):\n            exact += 1\n        else:\n            diff = np.abs(rebuilt.astype(int) - cached.astype(int))\n            differing.append((row["SeriesInstanceUID"], float(diff.mean()), int(diff.max())))\n\n    compared = exact + len(differing)\n    check(f"D1 all {compared} rebuilt series are byte-identical to the cache",\n          not differing,\n          f"{exact}/{compared} exact, {missing} series not found on disk\\n"\n          + "\\n".join(f"{uid[-12:]}  mean |diff| {m:.3f}  max {x}"\n                      for uid, m, x in differing[:10])\n          if differing else f"{exact}/{compared} exact, {missing} not found on disk")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--sample", type=int, default=2000,\n                        help="series to scan for blank/degenerate volumes")\n    parser.add_argument("--rebuild", type=int, default=40,\n                        help="series to rebuild from raw DICOM and compare byte for byte")\n    parser.add_argument("--expect-series", type=int, default=24371)\n    parser.add_argument("--expect-studies", type=int, default=4407)\n    args = parser.parse_args()\n\n    print(f"cache geometry from config: {config.CACHE_SLICES} x {config.CACHE_SIZE} "\n          f"x {config.CACHE_SIZE} {config.CACHE_DTYPE}, "\n          f"percentiles {config.CLIP_PCT_LOW}-{config.CLIP_PCT_HIGH}, "\n          f"MIRROR_IN_CACHE={config.MIRROR_IN_CACHE}")\n    print(f"data root: {config.DATA_ROOT}\\n")\n\n    manifest = load_manifests(args.split)\n    check_inventory(manifest, args.expect_series, args.expect_studies)\n    print()\n    check_sharding(manifest)\n    print()\n    by_shard = check_volumes(manifest, args.sample)\n    print()\n    check_rebuild(manifest, by_shard, args.rebuild, args.split)\n\n    print("\\n" + "=" * 70)\n    if FAILURES:\n        print(f"{len(FAILURES)} check(s) FAILED: {\', \'.join(FAILURES)}")\n        raise SystemExit(1)\n\n    # A manifests-only run proves the bookkeeping, not the pixels. Saying\n    # otherwise would be the exact overclaim this script exists to prevent.\n    if not by_shard:\n        print("manifest checks passed: the bookkeeping is sound (inventory, "\n              "sharding, zero build errors).")\n        print("NOT verified: volume contents and the byte-for-byte rebuild. "\n              "Attach the cache shards")\n        print("and the competition data to settle whether the shipped pixels "\n              "match today\'s code.")\n        return\n    print("all checks passed -- the shipped cache matches the code that reads it")\n\n\nif __name__ == "__main__":\n    main()\n',
    'tools/audit_orientation.py': '"""Corpus-wide audit of the two mirroring rulers, header-only.\n\nFour of the twelve findings are compartment-specific, so mirroring a knee along\nthe wrong axis does not add noise -- it *transposes* Medial/Lateral Meniscus and\nMedial/Lateral OA for that study. Two independent implementations decide that\naxis:\n\n    cache.mirror_axis          from ImageOrientationPatient (scanner geometry)\n    dataset.PLANE_MIRROR_AXIS  from the Anatomical_Plane string in the CSV\n\nOnly the second one runs in production (`MIRROR_IN_CACHE` is False, so\n`canonicalize_side` is never reached). The first is the better definition --\ncontinuous, and it handles obliquely-prescribed series -- and it is dead code.\nNobody has ever checked that they agree across the corpus.\n\nThey can only diverge once a series is tilted more than 45 degrees off its\nnominal plane, i.e. only when the CSV\'s plane label is flatly wrong. This\nscript measures whether that happens, and how close to the boundary the rest of\nthe corpus sits.\n\nDecodes **no pixel data**. Runs on a CPU kernel in minutes off the existing\nindex; only falls back to reading DICOM headers if no index is attached.\n\n    !python tools/audit_orientation.py --out /kaggle/working/orientation_audit.parquet\n\nExit status is non-zero if any series would be mirrored along different axes by\nthe two rulers.\n"""\n\nimport argparse\nimport glob\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nfrom src import config\nfrom src.data import cache, dicom_io\nfrom src.model import dataset as knee_dataset\n\n# Left-right in DICOM patient coordinates. The axis most aligned with this is\n# the one a mirror has to reflect through.\nLEFT_RIGHT = np.array([1.0, 0.0, 0.0])\n\nAXIS_NAMES = {0: "reverse slices", 1: "flip rows", 2: "flip columns"}\n\n\ndef find_one(pattern):\n    """Locate an input on Kaggle, where dataset directory names are unpredictable.\n\n    Deliberately shaped like notebook 08\'s `find_inputs`: shallow candidates\n    first, and **never** a recursive walk of `/kaggle/input/competitions`. The\n    competition mount holds ~820,000 DICOM files under 24,371 directories, and\n    a `**` glob across it costs minutes before returning nothing useful.\n    """\n    for candidate in (f"data/meta/{pattern}", f"data/{pattern}", pattern,\n                      f"/kaggle/working/{pattern}",\n                      f"/kaggle/input/competitions/*/{pattern}",\n                      f"/kaggle/input/*/{pattern}"):\n        hits = sorted(glob.glob(candidate))\n        if hits:\n            return hits[0]\n\n    for root in sorted(glob.glob("/kaggle/input/*")):\n        if os.path.basename(root) == "competitions":\n            continue\n        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))\n        if hits:\n            return hits[0]\n    return None\n\n\ndef obliquity_degrees(orientation):\n    """Angle between the slice normal and the patient left-right axis.\n\n    0 degrees is a textbook sagittal acquisition (normal points left-right);\n    90 degrees is a textbook coronal or axial one. `mirror_axis` switches its\n    answer at 45, so this is the margin to a flip -- a series at 44 degrees\n    agrees with a "Sagittal" label only barely.\n    """\n    normal = dicom_io.slice_normal(orientation)\n    cosine = abs(float(np.dot(normal, LEFT_RIGHT)))\n    return float(np.degrees(np.arccos(np.clip(cosine, 0.0, 1.0))))\n\n\ndef within_series_spread(orientations, distinct):\n    """Does a series\' orientation vary enough to change the mirror decision?\n\n    Scanners write per-slice orientation, and the six floats are not always\n    bit-identical down a stack -- 13 of the 24,371 series carry two distinct\n    tuples. That is not by itself a defect: what matters is whether the\n    *derived* axis differs, because that is the only thing the pipeline reads.\n    Reporting raw tuple inequality as a failure would be a false alarm, and an\n    audit that cries wolf stops being read.\n\n    Only computed when the tuples actually differ, so the common case costs\n    nothing.\n    """\n    if distinct <= 1 or not orientations:\n        return {"distinct_axes": 1, "obliquity_spread_deg": 0.0}\n\n    axes = {cache.mirror_axis(o) for o in orientations}\n    degrees = [obliquity_degrees(o) for o in orientations]\n    return {"distinct_axes": len(axes),\n            "obliquity_spread_deg": float(max(degrees) - min(degrees))}\n\n\ndef series_from_index(index_path):\n    """One row per series from the header index, plus within-series consistency."""\n    wanted = ["SeriesInstanceUID", "StudyInstanceUID", "ImageOrientationPatient",\n              "ImagePositionPatient", "ImageLaterality", "Laterality",\n              "SeriesDescription", "ProtocolName", "BodyPartExamined"]\n    index = pd.read_parquet(index_path)\n    present = [c for c in wanted if c in index.columns]\n    index = index[present]\n    print(f"index: {len(index):,} headers, {index[\'SeriesInstanceUID\'].nunique():,} series")\n\n    rows = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        raw = [h["ImageOrientationPatient"] for h in headers\n               if isinstance(h.get("ImageOrientationPatient"), (list, np.ndarray))\n               and len(h["ImageOrientationPatient"]) == 6]\n        orientations = [tuple(np.round(o, 4)) for o in raw]\n\n        row = {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "orientation": list(raw[0]) if raw else None,\n            "n_slices": len(headers),\n            "n_with_orientation": len(raw),\n            "distinct_orientations": len(set(orientations)),\n            "laterality_tag": dicom_io.resolve_laterality(headers),\n        }\n        row.update(within_series_spread(raw, row["distinct_orientations"]))\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef series_from_dicom(split):\n    """Fallback: read one header per series straight off the DICOM tree."""\n    root = Path(config.DATA_ROOT) / split\n    if not root.exists():\n        raise SystemExit(f"no index and no DICOM tree at {root}")\n\n    rows = []\n    for study_dir in sorted(p for p in root.iterdir() if p.is_dir()):\n        for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):\n            paths = sorted(series_dir.glob("*.dcm"))\n            if not paths:\n                continue\n            # Orientation is fixed within an acquisition; two headers is enough\n            # to notice when it is not.\n            probe = [dicom_io.read_header(p) for p in paths[:2]]\n            raw = [h["ImageOrientationPatient"] for h in probe\n                   if isinstance(h.get("ImageOrientationPatient"), (list, np.ndarray))\n                   and len(h["ImageOrientationPatient"]) == 6]\n            orientations = [tuple(np.round(o, 4)) for o in raw]\n            row = {\n                "SeriesInstanceUID": series_dir.name,\n                "StudyInstanceUID": study_dir.name,\n                "orientation": list(raw[0]) if raw else None,\n                "n_slices": len(paths),\n                "n_with_orientation": len(raw),\n                "distinct_orientations": len(set(orientations)),\n                "laterality_tag": dicom_io.resolve_laterality(probe),\n            }\n            row.update(within_series_spread(raw, row["distinct_orientations"]))\n            rows.append(row)\n        if len(rows) % 500 == 0 and rows:\n            print(f"  scanned {len(rows)} series", flush=True)\n    return pd.DataFrame(rows)\n\n\ndef audit(series, series_meta):\n    """Attach both rulers\' answers to every series and compare them."""\n    frame = series.merge(\n        series_meta[["SeriesInstanceUID", "Anatomical_Plane"]],\n        on="SeriesInstanceUID", how="left")\n\n    frame["geometry_axis"] = [\n        cache.mirror_axis(o) if o is not None else None for o in frame["orientation"]\n    ]\n    frame["plane_axis"] = [\n        knee_dataset.PLANE_MIRROR_AXIS.get(str(p), knee_dataset.DEFAULT_MIRROR_AXIS)\n        for p in frame["Anatomical_Plane"]\n    ]\n    frame["obliquity_deg"] = [\n        obliquity_degrees(o) if o is not None else None for o in frame["orientation"]\n    ]\n    frame["has_geometry"] = frame["orientation"].notna()\n    frame["plane_known"] = frame["Anatomical_Plane"].isin(knee_dataset.PLANE_MIRROR_AXIS)\n    frame["axes_agree"] = (frame["geometry_axis"] == frame["plane_axis"]) | ~frame["has_geometry"]\n    return frame\n\n\ndef report(frame):\n    failures = []\n    total = len(frame)\n\n    # --- the question the script exists to answer -------------------------\n    comparable = frame[frame["has_geometry"]]\n    disagree = comparable[comparable["geometry_axis"] != comparable["plane_axis"]]\n    ok = len(disagree) == 0\n    if not ok:\n        failures.append("axis disagreement")\n    print(f"[{\'PASS\' if ok else \'FAIL\'}] the two mirroring rulers agree on every series "\n          f"with geometry")\n    print(f"         {len(comparable) - len(disagree):,}/{len(comparable):,} agree; "\n          f"{len(disagree):,} disagree")\n    if len(disagree):\n        affected = disagree["StudyInstanceUID"].nunique()\n        print(f"         {affected:,} studies affected -- these have Medial/Lateral "\n              f"Meniscus and Medial/Lateral OA transposed today")\n        print(disagree.groupby(["Anatomical_Plane", "geometry_axis"])\n              .size().to_string())\n\n    # --- how robust that agreement is -------------------------------------\n    print()\n    print("[INFO] obliquity: angle between the slice normal and patient left-right")\n    print("       (mirror_axis switches its answer at 45 degrees)")\n    for plane, group in comparable.groupby("Anatomical_Plane"):\n        deg = group["obliquity_deg"]\n        near = int(((deg > 35) & (deg < 55)).sum())\n        print(f"       {plane:<9} n={len(group):>6,}  median {deg.median():>5.1f}deg  "\n              f"range {deg.min():>5.1f}-{deg.max():>5.1f}  "\n              f"{near:>5,} within 10deg of the 45deg boundary")\n\n    # --- the branches that silently take a default ------------------------\n    print()\n    no_geom = frame[~frame["has_geometry"]]\n    ok = len(no_geom) == 0\n    print(f"[{\'PASS\' if ok else \'WARN\'}] every series carries ImageOrientationPatient")\n    print(f"         {len(no_geom):,}/{total:,} lack it. These sort by InstanceNumber "\n          f"(unreliable) and, if the geometry ruler were ever made live, would "\n          f"default to \'{AXIS_NAMES[2]}\' -- wrong for a sagittal series.")\n    if len(no_geom):\n        print(no_geom["Anatomical_Plane"].value_counts(dropna=False)\n              .to_string(header=False))\n\n    unknown_plane = frame[~frame["plane_known"]]\n    ok = len(unknown_plane) == 0\n    if not ok:\n        failures.append("unknown plane string")\n    print()\n    print(f"[{\'PASS\' if ok else \'FAIL\'}] every series has a recognised Anatomical_Plane")\n    print(f"         {len(unknown_plane):,}/{total:,} unrecognised or missing -- these "\n          f"silently take DEFAULT_MIRROR_AXIS = {knee_dataset.DEFAULT_MIRROR_AXIS} "\n          f"(\'{AXIS_NAMES[knee_dataset.DEFAULT_MIRROR_AXIS]}\') in the LIVE path")\n    if len(unknown_plane):\n        print(unknown_plane["Anatomical_Plane"].value_counts(dropna=False)\n              .to_string(header=False))\n\n    split_axis = frame[frame["distinct_axes"] > 1]\n    ok = len(split_axis) == 0\n    if not ok:\n        failures.append("mirror axis varies within a series")\n    print()\n    print(f"[{\'PASS\' if ok else \'FAIL\'}] the mirror axis is constant down every series")\n    print(f"         {len(split_axis):,}/{total:,} series would mirror different "\n          f"slices along different axes -- one axis per series is then the wrong model")\n\n    mixed = frame[frame["distinct_orientations"] > 1]\n    if len(mixed):\n        spread = mixed["obliquity_spread_deg"]\n        print(f"         ({len(mixed):,} series carry more than one orientation tuple, "\n              f"but only as float variation: obliquity spread max "\n              f"{spread.max():.3f}deg, far below the 45deg that would change the "\n              f"answer. Not a defect.)")\n\n    # --- the other half of the mirror decision ----------------------------\n    print()\n    sides = {}\n    for study, group in frame.groupby("StudyInstanceUID"):\n        sides[study] = dict(zip(group["SeriesInstanceUID"], group["laterality_tag"]))\n    coverage = dicom_io.laterality_coverage(sides)\n    print("[INFO] laterality from DICOM tags alone (before the classifier):")\n    print(f"       known before study propagation: {coverage[\'known_before\']:,}"\n          f"/{coverage[\'total_series\']:,} "\n          f"({100 * coverage[\'known_before\'] / coverage[\'total_series\']:.1f}%)")\n    print(f"       known after:                    {coverage[\'known_after\']:,}"\n          f"/{coverage[\'total_series\']:,} "\n          f"({100 * coverage[\'known_after\'] / coverage[\'total_series\']:.1f}%), "\n          f"recovered {coverage[\'recovered\']:,}")\n    print(f"       studies whose series disagree on the side: "\n          f"{coverage[\'conflicting_studies\']:,} "\n          f"(left unresolved by design -- a conflict breaks the one-knee-per-study "\n          f"assumption)")\n    unknown = coverage["total_series"] - coverage["known_after"]\n    print(f"       {unknown:,} series ({100 * unknown / coverage[\'total_series\']:.1f}%) "\n          f"have no side from metadata and rest entirely on the classifier")\n\n    return failures\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--index", default=None,\n                        help="header index parquet; auto-discovered if omitted")\n    parser.add_argument("--series-csv", default=None,\n                        help="train_series.csv; auto-discovered if omitted")\n    parser.add_argument("--out", default=None,\n                        help="write the per-series audit rows here as parquet")\n    parser.add_argument("--scan", action="store_true",\n                        help="read DICOM headers directly instead of using an index")\n    args = parser.parse_args()\n\n    csv_name = "test_series.csv" if args.split == "test_series" else "train_series.csv"\n    series_csv = args.series_csv or find_one(csv_name)\n    if not series_csv:\n        raise SystemExit(f"{csv_name} not found")\n    series_meta = pd.read_csv(series_csv)\n    print(f"series metadata: {series_csv}  ({len(series_meta):,} rows)")\n\n    if args.scan:\n        series = series_from_dicom(args.split)\n    else:\n        index_name = ("index_test.parquet" if args.split == "test_series"\n                      else "index_train.parquet")\n        index_path = args.index or find_one(index_name)\n        if index_path:\n            print(f"header index: {index_path}")\n            series = series_from_index(index_path)\n        else:\n            print(f"no {index_name} found; falling back to a DICOM header scan")\n            series = series_from_dicom(args.split)\n\n    print(f"series audited: {len(series):,}\\n")\n    frame = audit(series, series_meta)\n    failures = report(frame)\n\n    if args.out:\n        os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)\n        keep = [c for c in frame.columns if c != "orientation"]\n        frame[keep].to_parquet(args.out, index=False)\n        print(f"\\nper-series audit written to {args.out}")\n\n    print("\\n" + "=" * 70)\n    if failures:\n        print(f"FAILED: {\', \'.join(failures)}")\n        raise SystemExit(1)\n    print("the two mirroring rulers agree across the corpus")\n\n\nif __name__ == "__main__":\n    main()\n'
}

for path, source in FILES.items():
    target = pathlib.Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)

sys.path.insert(0, ".")

# Drop any previously imported copy before importing. Writing the files above
# does not affect a module already in sys.modules, so re-running this cell in a
# kernel that has already imported src/ silently keeps the OLD code -- the new
# file sits on disk unused, and the run looks fine until something calls a
# function that only exists in the new version.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

from src import config
from src.data import dicom_io, cache, index as index_mod

# Only some notebooks materialise the model package.
if pathlib.Path("src/model/side_classifier.py").exists():
    from src.model import side_classifier
if pathlib.Path("src/model/net.py").exists():
    from src.model import (augment as knee_augment, averaging, conditional,
                           embed, dataset as knee_dataset, net, splits)
    from src.model import artifacts, experiments
# Only the fine-tuning notebook materialises these; lightning imports its
# framework at module scope and fuse pulls in the two extractors.
if pathlib.Path("src/model/inspect.py").exists():
    from src.model import inspect as knee_inspect
if pathlib.Path("src/labels/fuse.py").exists():
    from src.labels import fuse
if pathlib.Path("src/model/lightning.py").exists():
    from src.model import lightning as knee_lightning

print("modules ready:", len(FILES), "files")
print("labels:", len(config.LABELS))
print("cache geometry:", config.CACHE_SLICES, "x", config.CACHE_SIZE)
print("data root:", config.DATA_ROOT)

## What is actually attached

Printed before anything runs, because the commonest failure here is a missing
input rather than a real defect -- and a missing shard makes the run *look*
healthy while checking a third of the corpus.

In [ ]:
import glob, os

def show(pattern, label):
    hits = []
    for root in sorted(glob.glob("/kaggle/input/*")):
        if os.path.basename(root) == "competitions":
            continue
        hits += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    print(f"{label}: {len(hits)}")
    for path in hits:
        size = os.path.getsize(path) / 1e9
        print(f"   {size:6.2f} GB  {path}")
    return hits

shards = show("cache_train_series_*.npy", "cache shards")
manifests = show("manifest_train_series_*.parquet", "manifests")
index = show("index_train.parquet", "header index (optional)")

raw = any((config.DATA_ROOT / m).exists() for m in ("train_series", "test_series"))
print()
print("competition data:", "attached" if raw else "MISSING")
print("data root:", config.DATA_ROOT)
print()
if len(shards) != 3:
    print("WARNING: expected 3 cache shards. The rebuild check will only cover")
    print("         what is attached, and the inventory check will fail on A1.")
if not raw:
    print("WARNING: no raw DICOM, so the byte-for-byte rebuild will skip. That")
    print("         is the only check that can prove the pixels are correct.")

## The verification

Two settings:

- **`SAMPLE`** -- how many series to scan for blank or degenerate volumes.
  Reads one mid slice per series, ~50 KB each, so the full corpus is about
  1.2 GB of reads. Left at the full 24,371 rather than a sample: this closes
  the check instead of estimating it.
- **`REBUILD`** -- how many series to re-derive from raw DICOM and compare
  byte for byte. This is the decisive one. Roughly 0.55 s/series, so 500 is
  about five minutes. Raise it if you want tighter coverage; there is no
  runtime cap worth respecting on a CPU kernel.

A mismatch here means the cache and the code have diverged, and every
measurement this project has taken was taken on the cache.

In [ ]:
import subprocess, sys

SAMPLE = 24371     # scan every series for blank/degenerate volumes
REBUILD = 500      # series re-derived from raw DICOM and compared byte for byte

command = [sys.executable, "tools/verify_cache.py",
           "--sample", str(SAMPLE), "--rebuild", str(REBUILD)]
print(" ".join(command), flush=True)

result = subprocess.run(command, text=True, capture_output=True)
report = result.stdout + result.stderr
print(report)

# Persist the receipt. Save & Run All captures /kaggle/working, so the result
# survives the session rather than living in a scrollback nobody can cite.
with open("/kaggle/working/cache_verification.txt", "w") as handle:
    handle.write(report)
print("exit status:", result.returncode, "(0 = every check passed)")

## Orientation audit (optional)

Runs only if `index_train.parquet` is attached. It decodes no pixels and
answers a different question: do the two implementations of *which axis a left
knee reflects through* agree across all 24,371 series? They diverge only past
45 degrees of obliquity, i.e. only when the CSV's plane label is flatly wrong,
and a wrong answer transposes Medial/Lateral Meniscus and Medial/Lateral OA
rather than adding noise.

This already ran locally on 2026-09-02 and passed 24,371/24,371. It is here so
the result can be reproduced where the index lives natively.

In [ ]:
if index:
    command = [sys.executable, "tools/audit_orientation.py",
               "--index", index[0],
               "--out", "/kaggle/working/orientation_audit.parquet"]
    print(" ".join(command), flush=True)
    audit = subprocess.run(command, text=True, capture_output=True)
    print(audit.stdout + audit.stderr)
    print("exit status:", audit.returncode)
else:
    print("index_train.parquet not attached -- skipping the orientation audit.")
    print("Attach the dataset pssv27/knee-index to run it.")

## Reading the result

`exit status: 0` on the verification cell means the shipped cache reproduces
from raw DICOM and carries no blank volumes -- step 1 is then verified end to
end, artifact as well as code.

A **FAIL on D1** is the one that matters. It means the shards and `src/data/`
have drifted apart, and the fix is a cache rebuild, not a code change. Read the
per-series `mean |diff|` it prints: a max of 1 or 2 levels out of 255 is
percentile drift from a differently-sized decode, while a large difference
means different *slices*, which is a train/test skew.

`cache_verification.txt` and `orientation_audit.parquet` are written to
`/kaggle/working`. **Save & Run All** to keep them; Quick Save does not capture
the working directory.